# Tiny GPT Decoder (Code from Scratch)

## Series Notes
This is a series of notebook that attempt to show learners how to build a tiny decoder only GPT (Generative Pre-Train) LLM (Large Language Model). 

## This Notebook

This notebook gives a  detail explanation of LLM architecture with the focus on GPT. 

## How LLM is Build

Modern building of GPT LLM is not just feeding a large corpus of text into a transformer network. The building of LLM involves 3 main stage, that is:

- Stage 1 - Pre-Training
- Stage 2 - Post-Training
- Stage 3 - System Level Enhancement

![llm_build_stage.jpeg](assets/llm_build_stage.jpeg)

### Pre-Training

This is where we prepared a large corpus of text and feed into a transformer network to train the model. Our Tiny GPT project focus on pre-training using the transformer network. The resulting model is called foundational LLM or base LLM.

More will be discussed in this project.


### Post-Training

In post training, we further enhance the model using Supervised Fine-Tuning (SFT).

There are several layers of fine-tuning each with different purpose.
- Instructional fine-tuning
- Preference Alignment (RLHF/DPO)
- Reasoning Training
- We also add safety moderation at this stage

### System Level Enhancement

At this stage, LLM vendor will add additional capabilities to enhance the service delivery. This only happens on the recent LLM where frontier models has added more capabilities such as projects, web search capabilities, connecting external API, agentic AI etc.

## GPT Architecture

### Training Stage

Let us look at only the pre-training stage. In pre-training, we took a large corpus of the text and feed into the transformer network. The transformer network will make prediction and the training model will compare loss between actual and prediction text. Then, the model will perform back propagation to adjust the model weights. The process of back propagation will improve the model weights so that it makes the prediction better.

```text

Text Processing (Large Corpus) -> Transformer Network -> Compare Loss -> Back Propagation

```

![pre_training_stage.jpeg](assets/pre_training_stage.jpeg)

### Inference Stage

The inference process is similar but it will make prediction instead of comparing loss. During inference, we do not process a large corpus of text and instead we only process the query text. We feed the query into the transformer network and the model will make the prediction of the next token after the query. There is also a decoding process to select the best possible token during the prediction. Then we perform auto regression to keep making next token prediction until we get a stop signal or when we reach the max token size.

```text
Text Processing -> Transformer Network -> Predict Next Token -> Repeat Next Token Prediction until EOS or max token reached
```

![llm_response_generation.jpeg](assets/llm_response_generation.jpeg)

Example:

Prompt -> "mary"

The prediction loop as follows:

Prediction 1:
- Input -> "mary"
- Next Token Prediction 1: -> "had"

Prediction 2:
- Input -> "mary had"
- Next Token Prediction 2: -> "a"

Prediction 3:
- Input -> "mary had a"
- Next Token Prediction 2: -> "little"

Prediction 4:
- Input -> "mary had a little"
- Next Token Prediction 2: -> "lamb"

## Text Processing

Text Processing involves 2 stage, Stage 1 is the setup where we setup the vocabulary, token ID and embeddings initialization once. Stage 2 involves extracting a limited amount of token, look up the embeddings and feed it into the network. Stage 2 is the processing part of the training.

### Vocabulary and Embeddings Setup



![text_setup.jpeg](assets/text_setup.jpeg)

### Tokenization

Tokenization is the process of separating text into small element for prediction.

Traditionally, we tokenize the text into words. This means that we split the text into unique words. However, we improved the tokenization so that it is more efficient. We introduced sub-words. This is

Consider the words `cook`, `cooking`, `cooked`, `play`, `playing`, and `played`. A word-level tokenizer needs 6 separate vocabulary entries. If we split them into sub-words, we only need 4: `cook`, `play`, `ing`, and `ed`. This keeps the vocabulary small, and the savings grow as more words are added (e.g., `walk`, `walking`, `walked` need only one new entry, `walk`).

More importantly, sub-words let the model handle words it has never seen. If jumping isn’t in the vocabulary, a word-level model would treat it as unknown `UNK`, but a sub-word tokenizer can split it into jump + ing. The trade-off is that a single word may become multiple tokens, so sequences get slightly longer.

It also helps with grammar: the model already knows what `ing` means from `cooking`, so it automatically understands that `jumping` is an ongoing action without having to re-learn it from scratch.

In traditional NLP, we usually removed punctuation such as Word2Vec. But in modern LLM which uses transformer architecture, position is important and punctuation also provide useful information. Thus modern LLM based tokenization includes punctuation.


> In this exercise, we simplified the tokenization process into word tokenization.

#### Tokenization Process

In the tokenization process, we need to define the vocabulary size. Then tokenization algorithm such as BPE will combine each characters until we have reach the total vocabulary size.


> In this exercise we use the number of unique words in the corpus as our vocab size since our corpus is very small.


Next, we need to provide an ID to each unique token.

> In our exercise, since we use word tokenization, each unique word will be given an ID.

The exact step is as follows using word tokenization:

- Step 1 : Read the entire corpus and count the number of unique words. This will be the vocabulary size.
- Step 2 : Assign a token ID for each word/token. We also need reference table that can convert word to token id and vice versa.
- Step 3 : Convert the corpus into token IDs.
- Step 4 : to be continue....



### Embedding

Embedding is a number that represent a token. At the begining, we randomly initialized the embedding. During training the back propagation process will also adjust the embedding for the related token.

If our vocabulary size is 50, does it means that we have a vector of 50 by 1 so that each token has one embedding.

In real implementation, we do not use one embedding number as we want to capture more relationship. We can to caupture relationship between `bank` and `deposit` and at the same time also capture `river` and `bank`. We cannot rely on one embedding number to capture such complex relationship.

In GPT3 the vocabulary size is 50,257 tokens and the embedding dimension is 12,288

> In our TinyGPT example, our vocabulary size is 19. We define the embedding dimension to be 4. So our embedding matrix is 19 by 4.

Please note that we did not assigned meaning to the dimension, we just rely on the training process to capture the relationship.



#### Embedding Process

During initialization, the setup of embedding matrix is closely depend on tokenization and how we define the vocabulary size.

We use token ID to link the token with embedding.

We define the embedding matrix as vocab_size by dimension.
Then we randomly assign numbers to the embedding. During the training and back propagation, we will adjust the embeddings.

We need to setup the embeddings, thus we continue the previous precess

- Step 4 : Define the block size (context window)  and the depth of the embeddings (dimensions). The block size and the dimension formed a matrix of numbers to represent each word/token in the vocabulary for each training cycle.
- Step 5 : Randomly initialized the embeddings according to vocabulary size and dimension. This is our embedding matrix.


> Note: We do not need to add token id in the embeddings, but the index of the matrix will be used as token ID.

### Text Processing for Training

Once our embedding matrix and vocabulary with token ID is defined, we can start our training process. During the setup process we also convert all the corpus into ID. So the training process is as follows:

- Step 1 : Extract token ID according to our block size (context window).
- Step 2 : Refer to the embedding matrix and get the relevant embeddings (This will be block size by dimension, 5 by 4)
- Step 3 : Add position encoding, in our case, it will be 0,1,2,3,4
- The resulting input matrix (5,4) will be feed into the transformer network.  

![text_processing_training.jpeg](attachment:text_processing_training.jpeg)

#### Block Size

Block size is the number of token we send to the transformer network at one go. LLM user usually learn about the context windows which is the same as block size.

> In our example, we use block size of 5


GPT 3 uses context windows or block size of 2,048

#### Positional Encoding

The principal of transformer architecture is that the relative position of one word to another is important. Thus we need to add position information to the embeddings.

Our training corpus is the nursery rhyme "mary had a little lamb". IF we get the first block which is  

```text
mary had a little lamb
```

We need to define the position for each word.

> In this exercise, we use use 0,1,2,3,4 as the encoding for each block.

Please note that we start fresh encoding for each block.

Please note that this type of encoding does not yield effective result. Sine and cosine encoding are more popular. However, in modern LLM training, we use RoPE as position encoding method.



### Corpus

In [2]:
# Text corpus
corpus = """
mary had a little lamb
its fleece was white as snow
and everywhere that mary went
the lamb was sure to go
mary had a little lamb
the lamb was sure to go
"""


### Define Parameters

In [3]:
# Set Parameters

# d_model is embedding dimension
d_model = 4

# block_size is the same as context window or maximum sequence length
block_size = 5


## Text Processing - Setup

In [4]:
# Tokenization

# Split corpus into words
word_tokens = corpus.split()

# Use set to isolate unique words
vocab = sorted(list(set(word_tokens)))
vocab_size = len(vocab)

# Create token id and create lookup table from word to token id
# i is the index or token id in this case, so we assign token id using sequential number
words_to_id_ref = {word: i for i, word in enumerate(vocab)}

# Create lookup table from id to words
id_to_words_ref = {i: word for i, word in enumerate(vocab)}

In [5]:
vocab_size

19

In [6]:
# convert corpus to id
corpus_in_id = [words_to_id_ref[word] for word in word_tokens]


In [7]:
# Setup embeddings initialized randomly
random_seed = 42
embeddings = np.random.default_rng(random_seed).standard_normal((vocab_size, d_model))
embeddings.shape


(19, 4)

## Text Processing - Training

In [8]:
def get_sequence_block(block_size, start_position, corpus_in_id):
  """
  This function extract the block in id from the corpus in id
  User need to supply the block size and the start position
  Return: X, Y, next_start_position
  """
  X = corpus_in_id[start_position: start_position+block_size]
  Y = corpus_in_id[start_position+1: start_position+block_size+1]
  next_start_position = start_position + block_size
  return X, Y, next_start_position

In [9]:
def get_N(corpus_in_id, block_size):
  """
  This function calculate the number of block we need to run for the entire corpus
  If the remaining word could not feed into the network, we will just drop it
  """
  N = (len(corpus_in_id)-1) // block_size
  return N

In [10]:
N = get_N(corpus_in_id, block_size=block_size)
N

6

In [11]:
X, Y, next_start = get_sequence_block(block_size=block_size, start_position=0, corpus_in_id=corpus_in_id)
X, Y, next_start

([10, 6, 0, 9, 8], [6, 0, 9, 8, 7], 5)

In [12]:
# Get the embedding based on X input
# Since we define 5 token in one pass so the matrix is 5 by 4 where 4 is the dimension
X_input = [embeddings[x] for x in X]
X_input = np.array(X_input)
X_input.shape

(5, 4)

In [13]:
# Get positional Encoding
position = np.arange(0,block_size)
position = position.reshape(-1,1)
position

array([[0],
       [1],
       [2],
       [3],
       [4]])

In [14]:
X_input_pos = X_input + position
X_input_pos

array([[ 0.74325417,  0.54315427, -0.66550971,  0.23216132],
       [ 0.57167218,  0.64786645,  1.53230919,  1.36544406],
       [ 2.30471708,  0.96001589,  2.7504512 ,  2.94056472],
       [ 2.88605254,  2.15984352,  2.17551878,  3.65059279],
       [ 3.48775727,  3.18622727,  4.61597942,  5.12897229]])

In [15]:
X_input_pos.shape

(5, 4)

The X_input is ready to feed into the transformer network. Lets assign it to X for simplicity.

In [16]:
X = X_input_pos

## Transformer

Transformer is a neural network architecture that process language in parallel form resulting in increasing effectiveness in large language processing.

All frontier model such as ChatGPT, Claude, Gemini uses transformer network.

The core component of the Transformer network is the self attention mechanism. It allows the model to examine the word itself and its relation with other words to extract context.

Positional encoding is introduce to capture the relationship of words based on the position of words.

### Transformer Architecture

There are 3 main types of transformer architecture. they are encoder and decoder architecture, encoder only architecture and decoder only architecture.

The original transformer architecture uses encoder and decoder architecture.

#### Encoder Architecture
For the encoder architecture, we use mask language modeling (MLM) method to train the model. In mask language modeling, we mask certain words in a block so as to train the words. This allows the model to understand the context of the word given its position and relative position with other text. If the context windows is 512, all other unmask token are expose and the model can capture all the relationship between the unmask token to predict the mask token.

This type of training is bidirectional as the model can look at the left and right of the mask token to predict the mask token.  

Encoder only architecture is good to classification of text, name entity recognition and sentiment analysis. BERT is an example of encoder only architecture.


#### Decoder Architecture
For decoder architecture, we use causal language modeling (CLM). In this model we only train the model to predict next token and not next 2-3 tokens. The model can look back to capture the context but not looking forward.

Decoder only architecture is the most commonly used architecture as all the modern LLM uses decoder only architecture.

#### Encoder-Decoder Architecture
For encoder with decoder architecture, it is commonly use for language translation. This type of architecture also applied in text summarization, sequence-to-sequence tasks, and data-to-text generation


> For simplicity, this tiny project uses decoder only architecture


### Attention Formula

We will be implementing the attention formula so we need to understand it

$$\text{Attention (Q,K,V)} = \text{Softmax} \left( \frac{QK^T}{\sqrt{d_k}} \right) V$$


- $Q$ refers to the token we are asking or checking, in our first block `mary` will be the Q.

- $K$ refers to all other words in the block. In our context it will be `mary had a little lamb`. We will need to check `mary` against `mary`.

- $V$ refers to the value.

- $QK$ we multiple $Q$ by $K$ to get the relative position between $Q$ and each $K$. The $T$ refers to transpose, it is a necessary step in doing matrix multiplication.

- $d_k$ is a normalization factor so that the numbers $QK$ are stable. In our case $d_k$ is the model dimension.




### Attention Process

In computation, QKV also need to have weights. The size of the weights depends on the dimension of the model.

> In our tiny example our dimension is 4. So the matrix of the weights is 4 by 4.

Therefore WQ is 4 by 4. To get Q, we use input X and do a matrix multiplication with WQ to get Q.


> In our example, our input X, the shape of the matrix is (5, 4) WQ is (4,4); therefore Q is (5,4). The same applies to K and V.

Then we need to get the attention score, which is $QK^T$. After that we need to divide by the square root of the model dimension number.

Next we need to applied CLM, if we expose all embeddings, it wil become a MLM training. To applied CLM. Causal Language Masking allows us to mask all token from the first token to get the first token to predict second token.



### Causal Language Modeling (CLM)

CLM (Causal Language Modeling) masking is used in decoder-based transformers (e.g., GPT). The goal is to train the model to predict the next word using only the words that came before it. To do this, each position is prevented from attending to the words that come after it.

Consider the sentence `mary had a little lamb`. We want these training examples:

| Model sees              | Should predict |
|-------------------------|----------------|
| `mary`                  | `had`          |
| `mary had`              | `a`            |
| `mary had a`            | `little`       |
| `mary had a little`     | `lamb`         |

Note that the future words are hidden from the attention mechanism.

We could run each example separately, but that is slow. Instead, we feed the whole sentence in once and apply a causal mask, so each position can only see itself and earlier positions:

```text
          mary  had   a   little  lamb
mary       ✓     ✗    ✗     ✗      ✗
had        ✓     ✓    ✗     ✗      ✗
a          ✓     ✓    ✓     ✗      ✗
little     ✓     ✓    ✓     ✓      ✗
lamb       ✓     ✓    ✓     ✓      ✓

(✓ = allowed to attend, ✗ = blocked)
```

This is a lower-triangular matrix. It is applied to the attention scores before the softmax, by setting the blocked positions to -inf so they get zero weight.

The advantage is that one forward pass produces all the predictions
(`had`, `a`, `little`, `lamb`) in parallel. The targets are simply the input sequence shifted one position to the left. This is much more efficient than processing each prefix separately.

### Simple Architecture

In this architecture, we use the simplest architecture

```text

input text pre-processing (X)
      |
      V
Self attention
      |
      V
Residual layer
      |
      V
Normalized Layer
      |
      V
Feed Forward Network
      |
      V
Residual Layer
      |
      V
Normalized Layer

```

In Feed Forward Network, we have 3 layers

```text
Linear Layer
      |
      V
ReLU Layer
      |
      V
Linear Layer

```

![transformer_network.jpeg](attachment:transformer_network.jpeg)

## Self Attention Layer

We need a one time setup of model weights.

In [17]:
# Create a random generator with a random seed
rnd = np.random.default_rng(random_seed)


In [18]:
# Initializing weights (4,4) for WQ, WK and WV
WQ = rnd.standard_normal((d_model, d_model))
WK = rnd.standard_normal((d_model, d_model))
WV = rnd.standard_normal((d_model, d_model))

In [19]:
WQ

array([[ 0.30471708, -1.03998411,  0.7504512 ,  0.94056472],
       [-1.95103519, -1.30217951,  0.1278404 , -0.31624259],
       [-0.01680116, -0.85304393,  0.87939797,  0.77779194],
       [ 0.0660307 ,  1.12724121,  0.46750934, -0.85929246]])

In [20]:
WK

array([[ 0.36875078, -0.9588826 ,  0.8784503 , -0.04992591],
       [-0.18486236, -0.68092954,  1.22254134, -0.15452948],
       [-0.42832782, -0.35213355,  0.53230919,  0.36544406],
       [ 0.41273261,  0.430821  ,  2.1416476 , -0.40641502]])

> Make sure WQ and WK are initialized with different random number

### Self Attention Process

In [21]:
# Calculate QKV
# Q = X @ WQ
Q = X @ WQ
K = X @ WK
V = X @ WV

In [22]:
Q.shape

(5, 4)

In [23]:
# Compute attention score
attention_score = Q @ K.T
attention_score.shape

(5, 5)

In [24]:
# Divide by sqrt(d_k)
stable_factor = np.sqrt(d_model)
attention_stable = attention_score / stable_factor
attention_stable.shape

(5, 5)

## Causal Maskign Test Code

The code below shows how we setup CLM masking.


In [25]:
# causal masking test code
test1 = np.ones((5,5))
test1


array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]])

In [26]:
# causal masking test code
test2 = np.triu(test1, k=1)
test2

array([[0., 1., 1., 1., 1.],
       [0., 0., 1., 1., 1.],
       [0., 0., 0., 1., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0.]])

In [27]:
# causal masking test code
test3 = np.where(test2 == 1, -np.inf, test2)
test3

array([[  0., -inf, -inf, -inf, -inf],
       [  0.,   0., -inf, -inf, -inf],
       [  0.,   0.,   0., -inf, -inf],
       [  0.,   0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.,   0.]])

## Causal Masking

In [28]:
causal_mask = np.triu(np.ones(attention_stable.shape), k=1)
causal_mask

array([[0., 1., 1., 1., 1.],
       [0., 0., 1., 1., 1.],
       [0., 0., 0., 1., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0.]])

In [29]:
# actual causal masking
causal_mask = np.triu(np.ones(attention_stable.shape), k=1)
causal_mask = np.where(causal_mask == 1, -np.inf, causal_mask)
causal_mask

array([[  0., -inf, -inf, -inf, -inf],
       [  0.,   0., -inf, -inf, -inf],
       [  0.,   0.,   0., -inf, -inf],
       [  0.,   0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.,   0.]])

In [30]:
attention_stable

array([[  0.17317566,   0.6973632 ,   1.41703151,   1.7922252 ,
          2.90286602],
       [  1.90950856,   6.83338331,  14.79181287,  18.90261579,
         27.8343683 ],
       [  4.50094045,  15.38035587,  33.67772925,  43.05011569,
         62.97758068],
       [  4.75215104,  16.89636912,  36.53703779,  46.70971808,
         68.97232816],
       [  7.45004245,  26.57101602,  57.50766986,  73.38991285,
        108.51867912]])

In [31]:
attention_stable_mask = attention_stable + causal_mask
attention_stable_mask

array([[  0.17317566,         -inf,         -inf,         -inf,
                -inf],
       [  1.90950856,   6.83338331,         -inf,         -inf,
                -inf],
       [  4.50094045,  15.38035587,  33.67772925,         -inf,
                -inf],
       [  4.75215104,  16.89636912,  36.53703779,  46.70971808,
                -inf],
       [  7.45004245,  26.57101602,  57.50766986,  73.38991285,
        108.51867912]])

In [32]:
attention_stable_mask.shape

(5, 5)

In this 5 by 5 matrix we have

| col | mary | had | a | little | lamb |
| --- | --- | --- | --- | --- | --- |
| mary | 0.17 | -inf | -inf | -inf | -inf |
| had | 1.9 | 6.8 | -inf | -inf | -inf |
| a | 4.5 | 15.3 | 33.67 | -inf | -inf |
| little | 4.75 | 16.8 | 36.53 | 46.7 | -inf |
| lamb | 7.45 | 26.5 | 57.5 | 73.38 | 108.5 |


So `mary` can only see `mary` and it need to predict the next token. The second last row `mary had a little` is given but `lamb` is masked. the model suppose to predict mask in the 4th row.

In [33]:
def softmax(input):
  softmax = np.exp(input) / np.sum(np.exp(input), axis=-1, keepdims=True)
  return softmax

In [34]:
attention_softmax = softmax(attention_stable_mask)
attention_softmax.shape

(5, 5)

In [35]:
attention_softmax

array([[1.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [7.21841885e-03, 9.92781581e-01, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [2.13148176e-13, 1.13123206e-08, 9.99999989e-01, 0.00000000e+00,
        0.00000000e+00],
       [5.99851259e-19, 1.12774386e-13, 3.81983405e-05, 9.99961802e-01,
        0.00000000e+00],
       [1.27775857e-44, 2.57384336e-36, 7.01778672e-23, 5.54332898e-16,
        1.00000000e+00]])

In [36]:
attention_output = attention_softmax @ V
attention_output.shape

(5, 4)

In [37]:
attention_output

array([[-9.10170527e-01, -1.37187796e+00,  6.55224012e-01,
         1.08989024e+00],
       [ 9.18271066e-01,  1.10588032e-01, -7.07926203e-03,
         1.72334365e+00],
       [ 1.09744056e+00, -5.45092191e-01,  1.36018396e+00,
         4.52258668e+00],
       [ 3.18497431e-01, -2.18314948e+00,  1.73038717e+00,
         5.98472374e+00],
       [ 1.87968344e+00, -1.88633455e+00,  9.18957118e-01,
         8.22898507e+00]])

### Residual Layer

In [38]:
attention_output_residual = attention_output + X
attention_output_residual.shape

(5, 4)

### Normalized Layer

In [39]:
attention_output_residual_scaled = stats.zscore(attention_output_residual, axis=1)
attention_output_residual_scaled.shape

(5, 4)

### Feed Forward Layer (FFN)

In [40]:
W1 = rnd.standard_normal((d_model, d_model))
W2 = rnd.standard_normal((d_model, d_model))


In [41]:
# FFN 1 - Linear Layer
ffn_1 = attention_output_residual_scaled @ W1
ffn_1.shape

(5, 4)

In [42]:
# ReLU
ffn_1_relu = np.maximum(ffn_1, 0)
ffn_1_relu.shape

(5, 4)

In [43]:
# FFN 2 - Linear Layer
ffn_output = ffn_1_relu @ W2
ffn_output.shape

(5, 4)

### Second Residual Layer

In [44]:
seond_residual = ffn_output + attention_output_residual_scaled
seond_residual.shape

(5, 4)

### Second Normalized Layer

In [45]:
# 2nd layer
second_norm = stats.zscore(seond_residual, axis=1)
second_norm.shape

(5, 4)

### Convert to Logits

In [46]:
embeddings.shape

(19, 4)

In [47]:
second_norm

array([[-1.35010994, -0.49547167,  0.58468929,  1.26089231],
       [-1.34776201, -0.44391099,  0.45912462,  1.33254838],
       [-1.28442025, -0.66484959,  0.90977659,  1.03949325],
       [-1.34569661, -0.52392592,  0.64597045,  1.22365209],
       [-1.34743186, -0.46852504,  0.51044734,  1.30550955]])

In [48]:
logits = second_norm @ embeddings.T
logits.shape

(5, 19)

The output projection calculates a **next-token prediction score (logit)** for every word in the vocabulary. Each score represents how strongly the model favors that word as the next token, given the current context. Softmax converts these scores into probabilities.

In [49]:
logits

array([[ 1.72861270e+00,  2.95530411e+00,  1.94022896e+00,
        -1.45779233e+00,  4.27914353e-01,  1.10692761e+00,
         1.52478294e+00, -3.09411447e-02,  2.87845438e+00,
         9.08377428e-01, -1.36897837e+00,  5.25552311e-01,
         1.49400246e-02,  1.04513056e+00,  3.45423769e-01,
         3.27230845e+00, -2.14018679e+00, -3.23840916e+00,
         1.94893508e+00],
       [ 1.64887287e+00,  2.84486902e+00,  1.84151818e+00,
        -1.51979216e+00,  2.65459689e-01,  9.06803388e-01,
         1.46496807e+00, -3.05796051e-01,  2.83884546e+00,
         1.01493549e+00, -1.23902758e+00,  4.43703318e-01,
         2.89509367e-02,  1.03850951e+00,  5.99958776e-01,
         3.27266065e+00, -2.08695412e+00, -3.16728424e+00,
         2.00400405e+00],
       [ 1.96050182e+00,  3.15927678e+00,  2.19729081e+00,
        -1.30215668e+00,  9.11177605e-01,  1.64176383e+00,
         1.64842785e+00,  7.09401894e-01,  2.93293413e+00,
         6.31127214e-01, -1.67990162e+00,  7.29962925e-01,
    

### Calculate Loss

In [50]:
Y

[6, 0, 9, 8, 7]

In [51]:
logits_softmax = softmax(logits)
logits_softmax.shape

(5, 19)

In [52]:
logits_softmax

array([[0.05458664, 0.18613655, 0.06745127, 0.00225553, 0.01486621,
        0.02931514, 0.04452093, 0.00939554, 0.17236784, 0.02403603,
        0.00246502, 0.01639094, 0.00983666, 0.02755839, 0.0136891 ,
        0.25556707, 0.00113996, 0.00038013, 0.06804107],
       [0.05247517, 0.17352752, 0.06362366, 0.00220709, 0.01315664,
        0.02498487, 0.04366013, 0.00743108, 0.17248541, 0.02783802,
        0.0029225 , 0.01572372, 0.01038558, 0.02850207, 0.01838299,
        0.26616803, 0.00125171, 0.00042494, 0.07484888],
       [0.06408436, 0.21250708, 0.08120602, 0.00245356, 0.0224407 ,
        0.04659356, 0.0469051 , 0.01834029, 0.16946267, 0.01695946,
        0.00168169, 0.01872129, 0.00904319, 0.02433192, 0.00591796,
        0.20878433, 0.00096161, 0.00032833, 0.04927688],
       [0.05594704, 0.1919532 , 0.0696336 , 0.00228181, 0.01590148,
        0.03181872, 0.04496201, 0.01057924, 0.1722502 , 0.02244172,
        0.00227803, 0.01675697, 0.00962261, 0.02702215, 0.01178053,
        0.248

In [53]:
# compute loss

selected_probs = logits_softmax[[0,1,2,3,4],Y]
token_loss = -np.log(selected_probs)
loss = np.mean(token_loss)
loss

np.float64(3.3408421186997854)

This cannot be continue with back propagation as we are using pytorch back propagation fucntion.

## END